In [1]:
%matplotlib inline

In [ ]:

import warnings
import os
import pandas as pd
import seaborn as sns
import numpy as np
import networkx as netx
import pywt
import matplotlib.pyplot as plt
import matplotlib.style as style
import matplotlib.cm as cm
from scipy.io import loadmat
import scipy
from sklearn.manifold import LocallyLinearEmbedding
import nilearn as nl             # pip: nilearn==0.10.0
from nilearn.maskers import NiftiMasker
import nibabel as nib
import nltools as nlt
from matplotlib.colors import ListedColormap
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster



In [ ]:

################################################################################
# Archetypal Analysis        ####################################################
################################################################################
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class ArchetypalAnalysis(BaseEstimator, TransformerMixin):

    def __init__(self, n_archetypes=None, tmax=20, iterations=10):
        '''
        Implements archetypal analysis
            Archetypal Analysis as an Autoencoder (Bauckhage et al. 2015)

        Parameters
        ----------

        n_archetypes : the number of archetype

        tmax : the number of iterations of the derivative update

        iterations : the number of iterations of the overall algorithm


        Notes
        ---------

        The matrices have the following dimensions (following the above paper)
        [X] - m x n
        [Z] - m x k
        [B] - n x k
        [A] - k x n
        [e_A] - k x 1
        [e_B] - n x 1

        Source: https://miller-blog.com/archetypal-analysis/

        '''
        self.n_archetypes = n_archetypes
        self.tmax = tmax
        self.iterations = iterations

        N = self.n_archetypes
        x, y = np.zeros(N), np.zeros(N)
        x[1:] = np.cumsum(np.cos(np.arange(0, N - 1) * 2 * np.pi / N))
        y[1:] = np.cumsum(np.sin(np.arange(0, N - 1) * 2 * np.pi / N))
        self.map2D = np.vstack((x, y))

    def fit(self, X, y=None):
        """Fit the model with X using Archetypal Analysis
        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Training data, where n_samples is the number of samples
            and n_features is the number of features.
        y : Ignored
        Returns
        -------
        self : object
            Returns the instance itself.

        Source: https://miller-blog.com/archetypal-analysis/
        """
        self._fit(X)
        return self

    def _fit(self, X):
        k = self.n_archetypes
        m, n = X.shape

        B = np.eye(n, k)
        Z = X @ B

        for i in range(self.iterations):
            A = self._computeA(X, Z, self.tmax)
            B = self._computeB(X, A, self.tmax)
            Z = X @ B
            print('RSS = {}'.format(self._rss(X, A, Z)))

        self.Z_ = Z
        self.A_ = A

    def _computeA(self, X, Z, tmax):
        '''
        Algorithm 1 of Bauckhage et al. 2015
        Source: https://miller-blog.com/archetypal-analysis/
        '''
        m, n = X.shape
        k = self.n_archetypes

        A = np.zeros((k, n))
        A[0, :] = 1.0
        for t in range(tmax):
            # brackets are important to get reasonable sizes
            # [G] ~  k x n
            G = 2.0 * ((Z.T @ Z) @ A - Z.T @ X)
            # Get the argument mins along each column
            argmins = np.argmin(G, axis=0)
            e = np.zeros(G.shape)
            e[argmins, range(n)] = 1.0
            A += 2.0 / (t + 2.0) * (e - A)
        return A

    def _computeB(self, X, A, tmax):
        '''
        Algorithm 2 of Bauckhage et al. 2015
        Source: https://miller-blog.com/archetypal-analysis/
        '''
        k, n = A.shape
        B = np.zeros((n, k))
        B[0, :] = 1.0
        for t in range(tmax):
            # brackets are important to get reasonable sizes
            t1 = X.T @ (X @ B) @ (A @ A.T)
            t2 = X.T @ (X @ A.T)
            G = 2.0 * (t1 - t2)
            argmins = np.argmin(G, axis=0)
            e = np.zeros(G.shape)
            e[argmins, range(k)] = 1.0
            B += 2.0 / (t + 2.0) * (e - B)
        return B

    def archetypes(self):
        '''
        Source: https://miller-blog.com/archetypal-analysis/
        '''
        return self.Z_

    def transform(self, X):
        '''
        Source: https://miller-blog.com/archetypal-analysis/
        '''
        return self._computeA(X, self.Z_, self.tmax)

    def inverse_transform(self, X):

        return np.dot(self.Z_, X.T).T

    def _rss(self, X, A, Z):
        '''
        Source: https://miller-blog.com/archetypal-analysis/
        '''
        return np.linalg.norm(X - Z @ A)

    def score(self, X):

        return self._rss(X, self.A_, self.Z_)

def archetypal_plot(ax, data, dp, epsilon=0.2):
    '''
    Source: Dr. Luke Bovard
    '''
    ax.scatter(data[0, :], data[1, :], alpha=0.6, linewidths=10)
    ax.scatter(dp[0, :], dp[1, :], c='orange')

    for i in range(dp.shape[1]):
        if dp[0, i] < 0.5:
            eps_x = -epsilon
        else:
            eps_x = epsilon
        if dp[1, i] < np.max(dp[1, :]) / 2.0:
            eps_y = -epsilon
        else:
            eps_y = epsilon
        ax.text(dp[0, i] + eps_x, dp[1, i] + eps_y, "{}".format(i + 1))
    return ax

In [ ]:

def nii2cmu(nifti_file, mask_file=None):
    '''
    inputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename of a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output.  If ignored or set to None, no voxels will
                 be masked out.

    outputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.
    '''
    def fullfact(dims):
        '''
        Replicates MATLAB's fullfact function (behaves the same way)
        '''
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        else:
            aftervals = np.asmatrix(fullfact(dims[1:]))
            inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
            row = 0
            for i in range(aftervals.shape[0]):
                inds[row:(row + len(vals)), 0] = vals
                inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
                row += len(vals)
            return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        if type(nifti_file) == str:
            img = nib.load(nifti_file)
        elif type(nifti_file) == nib.nifti1.Nifti1Image:
            img = nifti_file
        else:
            raise ValueError('nifti_file must be a filename or nibabel Nifti1Image object')

        mask = NiftiMasker(mask_strategy='background')
        if mask_file is None:
            mask.fit(nifti_file)
        else:
            mask.fit(mask_file)

    hdr = img.header
    S = img.get_sform()
    vox_size = hdr.get_zooms()
    im_size = img.shape

    if len(img.shape) > 3:
        N = img.shape[3]
    else:
        N = 1

    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1]-1

    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]

    return {'Y': Y, 'R': R}

def cmu2nii(Y, R, template=None):
    '''
    inputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.

      template: a filename of a .nii or .nii.gz file to be used as an image
                template.  Header information of the outputted nifti images will
                be read from the header file.  If this argument is ignored or
                set to None, header information will be inferred based on the
                R array.

    outputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename for a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output

    outputs:
      img: a nibabel Nifti1Image object containing the fMRI data
    '''
    Y = np.array(Y, ndmin=2)
    img = nib.load(template)
    S = img.affine
    locs = np.array(np.dot(R - S[:3, 3], np.linalg.inv(S[0:3, 0:3])), dtype='int')

    data = np.zeros(tuple(list(img.shape)[0:3]+[Y.shape[0]]))

    # loop over data and locations to fill in activations
    for i in range(Y.shape[0]):
        for j in range(R.shape[0]):
            data[locs[j, 0], locs[j, 1], locs[j, 2], i] = Y[i, j]

    return nib.Nifti1Image(data, affine=img.affine)
    

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)

        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)

    return pd.DataFrame({'code': labels, 'Network': [list(lookup_table.values())[i - 1] for i in labels]})

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def purity(y_clust,y_class):

    size_clust = np.max(y_clust)
    len_clust = len(y_clust)
    clusters_labels_try = [None] * size_clust

    for i in range(len_clust):
        index = y_clust[i]
    
        if clusters_labels_try[index-1] is None:
            clusters_labels_try[index-1] = y_class[i]

        else:
            clusters_labels_try[index-1] = np.hstack((clusters_labels_try[index-1], y_class[i]))
    purity = 0
    for c in clusters_labels_try:
        c = c.astype(int)
        if isinstance(c, np.int64):
    
            c = np.array([c]) 
        y = np.bincount(c) #I find occurrences of the present elements
        maximum = np.max(y) #I take the item more frequently
        purity += maximum

    purity = purity/len_clust

    return purity

# Pie Man Data

## Load the data here

The data we're using for this, to start, is fMRI data from particpants scanned while being told a story in different listening conditions.  The story was told either intact, or scrambled by the paragraph, or scrambled by the word, or just while the participant was quietly resting.  

The fMRI data was then reduced using Hierarchicaly topographic factor analysis (HTFA which is a Bayesian factor analysis model designed to analyze brain network dynamics in multi-subject datasets. It builds upon Topographic Factor Analysis (TFA), extending it to incorporate data from multiple subjects, allowing for the identification of shared brain network hubs and the comparison of individual subject's brain activity against a global template. In this case, the >100,000 voxels was reduced to 700 nodes with varying widths.




In [ ]:
pieman_name = '../data/raw/pieman_data.mat'
pieman_data = loadmat(pieman_name)
pieman_conds = ['intact', 'paragraph', 'word', 'rest']

In [ ]:
posterior = loadmat('../data/raw/pieman_posterior_K700.mat')
centers = posterior['posterior']['centers'][0][0][0][0][0]
widths = np.array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T))

These lines of code separate the participant data by listening conditions (and skips over one participant with an incorrect number of timepoints in the paragraph condition)

In [ ]:
data = []
conds = []
for c in pieman_conds:
    if c == 'paragraph':
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.where(np.arange(pieman_data[c].shape[1]) != 3)[0]))
    else:
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.arange(pieman_data[c].shape[1])))
    data.extend(next_data)
    conds.extend([c]*len(next_data))


In [ ]:
conds_array = np.array(conds)

First, lets just play with some of the intact listening condition data:

In [ ]:
intact_list = [data[i] for i in np.where(conds_array=='intact')[0]]
intact_array = np.array(intact_list)

In [ ]:
paragraph_list = [data[i] for i in np.where(conds_array=='paragraph')[0]]
paragraph_array = np.array(paragraph_list)

In [ ]:
word_list = [data[i] for i in np.where(conds_array=='word')[0]]
word_array = np.array(word_list)

In [ ]:
rest_list = [data[i] for i in np.where(conds_array=='rest')[0]]
rest_array = np.array(rest_list)

## Network Analysis

Let's load in what we need for our network analysis:

In [ ]:
key = os.path.join('../data/raw/Schaefer2018_1000Parcels_7Networks_order.txt')
key = pd.read_csv(key, sep='\t', header=None, names=['id', 'name', 'x', 'y', 'z', 't']).drop('t', axis=1)
key['study'] = key['name'].apply(lambda x: x.split('_')[0])
key['hemisphere'] = key['name'].apply(lambda x: x.split('_')[1][0])
key['network'] = key['name'].apply(lambda x: x.split('_')[2])
key.drop('name', axis=1, inplace=True)

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode'
}

network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B'
}

colors = ['#888888']
colors.extend([v for k, v in network_colors.items()])
network_cmap = ListedColormap(colors, N=len(colors)*2, name='networks')  # why do I need to double the number of colors?

network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}

key['network'] = key['network'].apply(lambda x: lookup_table[x])
key['code'] = key['network'].apply(lambda x: network_codes[x])
key.set_index('id', inplace=True)
key.loc[0, 'code'] = 0  # append a background (non-network) code
key           # pip: nibabel==5.0.1

In [ ]:
nii_fname = os.path.join('../data/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz')


So this is a plot of the 7 networks in the brain determined by Yeo et al. (2017).  

In [ ]:
nx = nlt.Brain_Data(nii_fname)
nx.data = np.array([key.loc[i, 'code'] for i in nx.data]).astype(float)
networks = nx.to_nifti()
nl.plotting.plot_glass_brain(networks, cmap=network_cmap, display_mode='lyrz')
plt.show()
plt.clf()


We need a way to map our 700 nodes to these networks:

In [ ]:
# load network parcellations in CMU format
networks_cmu = nii2cmu(nii_fname)

# covert voxel values in the reference image to network codes
networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))

And these are the labels/code for each of those nodes:

In [ ]:
node_codes = node_labels(centers, widths, networks_cmu)
node_codes

Ok and plot that now (we could vary the node size relative to the width, but just for simplicity the nodes are all the same size here).

In [ ]:
nl.plotting.plot_connectome(np.eye(centers.shape[0]), centers, node_size=10, node_color=[colors[i] for i in node_codes['code']], display_mode='lyrz')
plt.show()
plt.clf()

Ok now that we have the network labels for each of those nodes, lets see the overall count for each of those nodes in this dataset:

In [ ]:
fig = plt.figure(figsize=(4, 3))
ax = plt.gca()

sns.histplot(data=node_codes, x='code', color='k', shrink=0.75, hue='code', hue_order=range(1, 8), palette=colors[1:], alpha=1.0, legend=False, ax=ax, discrete=True, edgecolor=None)
ax.set_xlabel('Network', fontsize=12)
ax.set_ylabel('Number of nodes', fontsize=12)

ax.set_xticks([])
ax.spines[['right', 'top']].set_visible(False)
plt.show()
plt.clf()
fig.savefig(os.path.join('htfa_node_counts.pdf'), bbox_inches='tight')

## Across all conditions!

In [ ]:
my_cmap = ListedColormap(network_colors.values())

In [ ]:
conds_listed = ['intact_list', 'paragraph_list', 'word_list', 'rest_list']

for u in conds_listed:
    
    num_archetypes=10
    n_node_clusters = 7
    data_list = eval(u)
    condition = u[:-5]
    
    
    # === CONCATENATE DATA FOR GROUP ANALYSIS ===
    group_data = np.hstack(data_list)
    
    # === RUN ARCHETYPAL ANALYSIS ===
    aa = ArchetypalAnalysis(n_archetypes=num_archetypes,iterations=10,tmax=300)
    aa.fit(group_data)
    archetypes = aa.archetypes()  # shape: (n_archetypes, n_features)
    
    
    coefficients = aa.transform(group_data)
    reshaped_coeffs = coefficients.reshape(np.array((num_archetypes, np.shape(data_list)[0], np.shape(data_list)[2])))
    averages_coeffs = reshaped_coeffs.mean(axis=1)

    
    for i in range(num_archetypes):

        df = pd.DataFrame(reshaped_coeffs[i, :, :])
        mean_weights = df.mean()
        similarity_matrix = df.corr()  # shape: (node x node)
            
        # Cluster subjects using hierarchical clustering
        dist = pairwise_distances(similarity_matrix, metric='euclidean')
        linkage_matrix = linkage(dist, method='average')
        order = leaves_list(linkage_matrix)
        
        # Reorder matrix
        sorted_matrix = similarity_matrix.iloc[order, :].T.iloc[order, :]
        
        # Define clusters (e.g., 3 clusters)
        #cluster_labels = fcluster(linkage_matrix, t=40, criterion='distance')
        cluster_labels = fcluster(linkage_matrix, n_node_clusters, criterion='maxclust')
        # Reorder cluster labels
        cluster_labels_ordered = cluster_labels[order]
        
        # Get indices where cluster boundaries change
        boundaries = np.where(np.diff(cluster_labels_ordered))[0] + 1
    
        # Plot with block outlines
        plt.figure(figsize=(6, 5))
        sns.heatmap(sorted_matrix, cmap="coolwarm")
        
        # Add block outlines
        for b in boundaries:
            plt.axhline(b, color='black', linewidth=1)
            plt.axvline(b, color='black', linewidth=1)
    
        print(len(boundaries))
        plt.title("Node-by-Node Similarity (Clustered with Blocks) for archetype: " + str(i + 1) + " Condition " + condition)
        plt.xlabel("Nodes (clustered)")
        plt.ylabel("Nodes (clustered)")
        plt.tight_layout()
        plt.show()
        plt.clf()

        # Optional: assign colors to Yeo networks
        node_networks = node_codes['Network']
        network_code = node_codes['code']
        # network_colors = colors
        

        corr_matrix = np.corrcoef(df.T)  # shape (n_nodes, n_nodes)
        
        # === Step 2: Threshold correlation to build edges ===
        threshold = 0.6  # only keep strong edges
        G = netx.Graph()
        
        # Add nodes with attributes (e.g., network label and color)
        for i in range(corr_matrix.shape[0]):
            net = network_code[i]
            G.add_node(i, network=node_networks[net], color=colors[net])
        # Add edges above threshold
        for i in range(corr_matrix.shape[0]):
            for j in range(i + 1, corr_matrix.shape[0]):
                if corr_matrix[i, j] > threshold:
                    G.add_edge(i, j, weight=corr_matrix[i, j])
        
        # === Step 3: Visualize (subsampled for clarity) ===
        # To make the plot readable, we’ll subsample a few nodes per network
        selected_nodes = []
        for net in range(1, 8):
            node_indices = np.where(network_code == net)[0]
            selected_nodes.extend(node_indices[:4])  # pick first 5 nodes per network
        
        H = G.subgraph(selected_nodes)
        
        pos = netx.spring_layout(H, seed=42)
        H_node_colors = [H.nodes[n]['color'] for n in H.nodes]
        
        plt.figure(figsize=(10, 8))
        netx.draw(H, pos, node_color=H_node_colors, with_labels=False,
                node_size=300, edge_color='gray', width=1.0)
        legend_handles = [plt.Line2D([0], [0], marker='o', color='w',
                                     label=name, markersize=10,
                                     markerfacecolor=color)
                          for name, color in network_colors.items()]
        plt.legend(handles=legend_handles, title="Yeo 7 Networks")
        plt.title("Node-Level Functional Connectivity Graph (Subsampled)")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        plt.clf()

        mean_weights_ordered = mean_weights[order]
        nodes_codes_ordered = node_codes['code'].iloc[order]
        for bb in range(1, 1+len(boundaries)):
            
            bb_inds = np.where(cluster_labels_ordered==bb)
            nl.plotting.plot_markers(nodes_codes_ordered.iloc[bb_inds], centers[order][bb_inds], node_vmin=1, node_vmax=7, alpha=mean_weights_ordered.iloc[bb_inds], title="Cluster " + str(bb) + " Condition " + condition, node_cmap=my_cmap)
            plt.show()
            plt.clf()
            hist_data = node_codes.iloc[order]
            sns.histplot(data=hist_data.iloc[bb_inds], x='code', color='k', binrange=(1,7), shrink=0.75, hue='code', hue_order=range(1, 8), palette=colors[1:], alpha=1.0, discrete=True, edgecolor=None)
            ax.set_xlabel('Network', fontsize=12)
            ax.set_ylabel('Number of nodes', fontsize=12)
            plt.show()
            plt.clf()

### Ok what I want to do with this analysis is to cluster based on subject
- find the participants that are most similar (is this consistent across archetypes?)
- Pull out the networks with the highest weights
- Find clusters that best distinguish conditions (rest vs other, and intact vs other, and each) 

In [ ]:
intact_all_list = eval('intact_list')
word_all_list = eval('word_list')
rest_all_list = eval('rest_list')
try_stack_intact = np.hstack(intact_all_list)
try_stack_word =  np.hstack(word_all_list)
try_stack_rest =  np.hstack(rest_all_list)
all_stack = np.hstack((try_stack_intact, try_stack_word, try_stack_rest[:300]))
group_data = all_stack
num_archetypes = 5

# === RUN ARCHETYPAL ANALYSIS ===
aa = ArchetypalAnalysis(n_archetypes=num_archetypes,iterations=100,tmax=300)
aa.fit(group_data)
archetypes = aa.archetypes()  # shape: (n_archetypes, n_features)


coefficients = aa.transform(group_data)
reshaped_coeffs = coefficients.reshape(np.array((num_archetypes, 36*3, 700)))
averages_coeffs = reshaped_coeffs.mean(axis=1)



In [ ]:
condition_labels = np.hstack((np.zeros(36), np.ones(36), np.ones(36)+1))
sampling_rate = .5
time_points = 300

for i in range(num_archetypes):
    df = pd.DataFrame(reshaped_coeffs[i, :, :])
    similarity_matrix = df.T.corr()  # shape: (subjects x subjects)
    
    
    # Cluster subjects using hierarchical clustering
    dist = pairwise_distances(similarity_matrix, metric='euclidean')
    linkage_matrix = linkage(dist, method='average')
    order = leaves_list(linkage_matrix)
    
    # Reorder matrix
    sorted_matrix = similarity_matrix.iloc[order, :].T.iloc[order, :]
    
    # Define clusters (e.g., 3 clusters)
    # cluster_labels = fcluster(linkage_matrix, t=3.5, criterion='distance')
    
    cluster_labels = fcluster(linkage_matrix, 3, criterion='maxclust') # so this is the for 3 clusters but run the line above if you want a distance metric for cluster instead
    # Reorder cluster labels
    cluster_labels_ordered = cluster_labels[order]
    
    # Get indices where cluster boundaries change
    boundaries = np.where(np.diff(cluster_labels_ordered))[0] + 1
    condition_labels_orders = condition_labels[order]
    condition_labels_orders_colors = ["black" if item == 2 else "green" if item == 1 else "purple" if item == 0 else item for item in condition_labels_orders]

    # Plot with block outlines
    plt.figure(figsize=(8, 7))
    ax = sns.heatmap(sorted_matrix, cmap="coolwarm", vmin=-1, vmax=1,
                xticklabels=order + 1, yticklabels=order + 1, square=True,
                cbar_kws={'label': 'Pearson Correlation'})

    ax.set_xticklabels(order + 1)
    for tick_label, color in zip(ax.get_xticklabels(), condition_labels_orders_colors):
        tick_label.set_color(color)
    ax.set_yticklabels(order + 1)
    for tick_label, color in zip(ax.get_yticklabels(), condition_labels_orders_colors):
        tick_label.set_color(color)
            
    # Add block outlines
    for b in boundaries:
        plt.axhline(b, color='black', linewidth=2)
        plt.axvline(b, color='black', linewidth=2)
        
    pure_cluster = purity(cluster_labels_ordered, condition_labels_orders)
    plt.title("Subject-by-Subject Similarity for Archetype: " + str(i + 1) + " Purity: " + str(np.round(pure_cluster,2)))
    plt.xlabel("Subjects (clustered)")
    plt.ylabel("Subjects (clustered)")
    plt.tight_layout()
    plt.show()
    plt.clf()


    plt.figure(figsize=(12, 8))

    plt.plot(archetypes[:, i], label='archetype: ' + str(i+1))
    plt.show()
    plt.clf()

    # Choose a wavelet (e.g., Morlet wavelet)
    wavelet = 'morl'
    
    # Define scales for the CWT
    scales = np.arange(1, 128)  # Adjust scales as needed to cover relevant frequencies/periods
    
    # Perform CWT for both time series
    coeff1, freqs1 = pywt.cwt(archetypes[:, i], scales, wavelet, sampling_period=1/sampling_rate)
    
    
    # Plot the wavelet power spectrum for each time series
    plt.figure(figsize=(12, 8))

    plt.imshow(np.abs(coeff1), extent=[0, time_points, freqs1[-1], freqs1[0]],
               cmap='jet', aspect='auto', vmax=abs(coeff1).max(), vmin=-abs(coeff1).max())
    # plt.colorbar(label='Magnitude')
    plt.ylabel('Frequency (Hz)')
    plt.title('Wavelet Power Spectrum for archetype ' + str(i+1) )
    

    plt.tight_layout()
    plt.show()
    plt.clf()

    nodes_codes_ordered = node_codes['code'].iloc[order]
    
    for bb in range(1, 2+len(boundaries)):
        
        bb_inds = np.where(cluster_labels_ordered==bb)
        reordered_df = df.iloc[order]
        cluster_bb = reordered_df.iloc[bb_inds]
        cluster_node_weights = cluster_bb.mean(axis=0)
        condition_labels_orders_clustered = condition_labels_orders[bb_inds]
        # top_cluster_node_weights = cluster_node_weights>.4
        sorted_items = sorted(cluster_node_weights.items(), key=lambda item: item[1], reverse=True)
        top_cluster_node_weights = [index for index, value in sorted_items[:20]]
        nl.plotting.plot_markers(node_codes['code'][top_cluster_node_weights], centers[top_cluster_node_weights], node_vmin=1, node_vmax=7, alpha=cluster_node_weights[top_cluster_node_weights], title="Archetype: " + str(i) + " Cluster: " + str(bb), node_cmap=my_cmap)
        plt.show()
        plt.clf()
        clustered_mean_weight_data = pd.DataFrame({'mean_weights': cluster_node_weights[top_cluster_node_weights],'network_code': network_code[top_cluster_node_weights]})
        ax = sns.barplot(x='network_code', y='mean_weights', data=clustered_mean_weight_data, hue='network_code', hue_order=range(1, 8), palette=colors[1:])
        sns.stripplot(x='network_code', y='mean_weights', data=clustered_mean_weight_data, jitter=0.2, color="black", alpha=0.5, ax=ax)
    
        plt.show()
        plt.clf() 

## Ok so this analysis looks at the archetypes across the intact and the word conditions.  It then attemps to cluster them and sees the corresponding nodes and weights that correpsond to those cluster
- modification I want to try: do a purity analysis and only view the clusters with the top purity